# BirdCLEF+ 2026 — Colab A100/H100 Training Pipeline

**Run order:** Cell 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9  
**Edit only Cell 0** before running the rest.

Critical constants (must match src/ and submission notebook exactly):
SR=32000, N_FFT=2048, HOP_LENGTH=512, N_MELS=128, F_MIN=20, F_MAX=16000, WINDOW_SECS=5, INPUT_H=224, INPUT_W=224, NUM_CLASSES=234

In [ ]:
# ============================================================
# CELL 0 — USER CONFIGURATION (edit only this cell)
# ============================================================

DRY_RUN = True   # Set False for full training. True = ~50 clips only, ~5 min total.

APPROACHES = {
    'approach3_b0':         True,   # Single-fold EfficientNet-B0 (uses existing src/train.py)
    'approach2_eat':        True,   # EfficientAT mn10 fine-tune
    'approach1_embeddings': True,   # Frozen embeddings + MLP heads
    'pseudo_labeling':      True,   # Pseudo-label all soundscapes
}

GOOGLE_DRIVE_CACHE = '/content/drive/MyDrive/birdclef2026_cache'

# Critical constants — must be identical in src/, both notebooks, and all config files
SR          = 32000
N_FFT       = 2048
HOP_LENGTH  = 512
N_MELS      = 128
F_MIN       = 20
F_MAX       = 16000
WINDOW_SECS = 5
INPUT_H     = 224
INPUT_W     = 224
NUM_CLASSES = 234

print('Cell 0 config loaded.')
print(f'DRY_RUN={DRY_RUN}')
print(f'Active approaches: {[k for k,v in APPROACHES.items() if v]}')

In [ ]:
# ============================================================
# CELL 1 — PACKAGE INSTALLS + sys.path
# ============================================================
import time as _t0_cell1; _t0_cell1 = _t0_cell1.time()
print(f"\n{'='*50}\n[CELL 1] Package installs\n{'='*50}")

# Core ML packages
!pip install -q \
    'kaggle' 'kagglehub' 'timm==0.9.16' \
    'onnxruntime' 'onnx' \
    'librosa' 'audiomentations' \
    'scikit-learn' 'tqdm' \
    'tensorflow' 'tensorflow-hub' \
    'opencv-python-headless'

# torch/torchaudio/torchvision already pre-installed on Colab GPU runtimes;
# install only if missing.
!python -c 'import torch' 2>/dev/null || pip install -q torch torchaudio torchvision

# EfficientAT — MobileNet-based audio model
!git clone -q https://github.com/fschmid56/EfficientAT /content/EfficientAT 2>/dev/null || echo 'EfficientAT already cloned'
!pip install -q -e /content/EfficientAT/ 2>/dev/null

# Add project root and EfficientAT to Python path
import sys, os
PROJECT_DIR = '/content/kaggle3'
for p in [PROJECT_DIR, '/content/EfficientAT']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Print installed versions
import subprocess as _sp
for pkg in ['torch', 'timm', 'tensorflow', 'onnxruntime']:
    try:
        v = _sp.run([sys.executable,'-c',f'import {pkg}; print({pkg}.__version__)'],
                    capture_output=True, text=True).stdout.strip()
        print(f'  {pkg}: {v}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')

import time as _t
print(f"[CELL 1] Done in {_t.time()-_t0_cell1:.1f}s")

In [ ]:
# ============================================================
# CELL 2 — KAGGLE CREDENTIALS + DATA DOWNLOAD + DRIVE MOUNT
# ============================================================
import time as _t0; _t0 = time.time() if 'time' in dir() else __import__('time').time()
import time, os, sys, glob, subprocess, shutil
from pathlib import Path
print(f"\n{'='*50}\n[CELL 2] Data download & Drive mount\n{'='*50}")

# ── Kaggle credentials ──────────────────────────────────────────────
# Tries KAGGLE_API_TOKEN first (single-secret style), then falls back
# to the classic KAGGLE_USERNAME + KAGGLE_KEY pair.
from google.colab import userdata

def _try_secret(key):
    try:
        return userdata.get(key)
    except Exception:
        return None

_token    = _try_secret('KAGGLE_API_TOKEN')
_username = _try_secret('KAGGLE_USERNAME')
_key      = _try_secret('KAGGLE_KEY')

if _token:
    os.environ['KAGGLE_API_TOKEN'] = _token
    print(f"Kaggle credentials: API-token style. Prefix: {_token[:8]}...")
elif _username and _key:
    os.environ['KAGGLE_USERNAME'] = _username
    os.environ['KAGGLE_KEY']      = _key
    print(f"Kaggle credentials: username/key style. User: {_username}")
else:
    print("WARNING: no Kaggle credentials found in Colab secrets. "
          "Add KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) via the key icon.")

# ── Download competition data ────────────────────────────────────────
import kagglehub
DATA_DIR = kagglehub.competition_download('birdclef-2026')
DATA_DIR = Path(DATA_DIR)
print(f"DATA_DIR: {DATA_DIR}")

# ── Download Perch v2 CPU model ──────────────────────────────────────
# Correct Kaggle model path (confirmed from Kaggle model registry):
#   Owner:     google
#   Model:     bird-vocalization-classifier   ← NOT 'perch'
#   Framework: TensorFlow2                   ← camelCase, NOT 'tfSavedModel'
#   Variation: perch_v2_cpu
#   Version:   1
PERCH_DIR = None
_perch_handles = [
    'google/bird-vocalization-classifier/TensorFlow2/perch_v2_cpu/1',  # canonical
    'google/bird-vocalization-classifier/TensorFlow2/perch_v2_cpu',    # latest version
    'google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1',  # lowercase variant
]
for _handle in _perch_handles:
    try:
        PERCH_DIR = Path(kagglehub.model_download(_handle))
        print(f"PERCH_DIR: {PERCH_DIR}  (handle: {_handle})")
        break
    except Exception as _e:
        print(f"  Perch handle '{_handle}' → {type(_e).__name__}: {str(_e)[:120]}")

if PERCH_DIR is None:
    print("WARNING: Perch model could not be downloaded automatically.\n"
          "  approach1_embeddings (Perch branch) will be skipped.\n"
          "  To fix manually: Runtime → Manage sessions → Add model →\n"
          "  search 'google bird-vocalization-classifier' → TensorFlow2 → perch_v2_cpu")

# ── Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(GOOGLE_DRIVE_CACHE, exist_ok=True)
print(f"Drive cache: {GOOGLE_DRIVE_CACHE}")

# ── Clone project from GitHub ─────────────────────────────────────────
# Repo: https://github.com/mohazed/birdclef_challenge
# IMPORTANT: src/ must be in that repo before running this cell.
# One-time push from your Mac:
#   cd /Users/mohorozovic/M2/S4/PDS/kaggle3
#   git add src/ configs/ notebooks/ && git commit -m "add src" && git push
GITHUB_REPO_URL = 'https://github.com/mohazed/birdclef_challenge.git'

if not Path(PROJECT_DIR + '/src').exists():
    print(f'Cloning {GITHUB_REPO_URL} → {PROJECT_DIR} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', GITHUB_REPO_URL, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('git clone stderr:', result.stderr[-500:])
        raise RuntimeError(f'git clone failed. Make sure src/ is pushed to {GITHUB_REPO_URL}')
    print(f'Cloned successfully.')
else:
    # Already present — pull latest changes
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'],
                   capture_output=True)
    print(f'Project already at {PROJECT_DIR} — pulled latest.')

# ── Verify src/ is present ────────────────────────────────────────────
src_files = list(Path(PROJECT_DIR + '/src').glob('*.py'))
if not src_files:
    raise FileNotFoundError(
        f"src/*.py not found in {PROJECT_DIR}.\n"
        "  The GitHub repo is missing the src/ directory.\n"
        "  Fix on your Mac:\n"
        "    cd /Users/mohorozovic/M2/S4/PDS/kaggle3\n"
        "    git add src/ && git commit -m 'add src' && git push\n"
        "  Then re-run this cell."
    )
print(f"src/ OK — {len(src_files)} Python files: {[f.name for f in src_files]}")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

# ── Validation block ─────────────────────────────────────────────────
required = [
    DATA_DIR / 'train_audio',
    DATA_DIR / 'train_soundscapes',
    DATA_DIR / 'train_soundscapes_labels.csv',
    DATA_DIR / 'taxonomy.csv',
    DATA_DIR / 'sample_submission.csv',
]
for p in required:
    assert Path(p).exists(), f"FAILED: required path missing: {p}"

import pandas as _pd, numpy as _np
ogg_clips     = list((DATA_DIR / 'train_audio').rglob('*.ogg'))
soundscapes   = list((DATA_DIR / 'train_soundscapes').glob('*.ogg'))
ann_df        = _pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
taxonomy_df   = _pd.read_csv(DATA_DIR / 'taxonomy.csv')
print(f"  Training clips  : {len(ogg_clips):,}  (expected ~35549)")
print(f"  Soundscapes     : {len(soundscapes):,}  (expected ~10658)")
print(f"  Annotation rows : {len(ann_df):,}  (expected ~1478)")
print(f"  Unique species  : {len(taxonomy_df):,}  (expected 234)")

if DRY_RUN:
    print('\nDRY_RUN=True: pipeline will process 50 clips only.')

print(f"[CELL 2] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 3 — BUILD folds.csv + MEL CACHE + SOUNDSCAPE WINDOWS
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 3] Build folds + mel cache + soundscape windows\n{'='*50}")

# ── Robust src/ path setup (safe to repeat; works even if Cell 1 didn't run) ──
import sys, os
from pathlib import Path
PROJECT_DIR  = '/content/kaggle3'
MEL_CACHE_DIR = '/content/mel_cache'
GOOGLE_DRIVE_CACHE = GOOGLE_DRIVE_CACHE if 'GOOGLE_DRIVE_CACHE' in dir() else '/content/drive/MyDrive/birdclef2026_cache'
DRY_RUN = DRY_RUN if 'DRY_RUN' in dir() else True
# ── Guarantee PROJECT_DIR is at sys.path[0] ─────────────────────────
while PROJECT_DIR in sys.path:
    sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
# Ensure src/__init__.py exists
(_src_init := Path(PROJECT_DIR) / 'src' / '__init__.py').parent.mkdir(parents=True, exist_ok=True)
_src_init.touch()
# Diagnose and fail fast if git clone didn't work
_dp = Path(PROJECT_DIR) / 'src' / 'data_prep.py'
if not _dp.exists():
    raise FileNotFoundError(
        f"src/data_prep.py not found at {_dp}. "
        "Run Cell 2 first to clone the repo, or check git clone output."
    )
print(f'sys.path[0]={sys.path[0]!r}  cwd={os.getcwd()!r}  src/data_prep.py=EXISTS')

from src.data_prep import (
    build_folds,
    build_mel_cache,
    extract_soundscape_annotation_windows,
)

FOLDS_CSV           = 'data/folds.csv'
MEL_CACHE_DIR       = '/content/mel_cache'   # fast SSD on Colab
SOUNDSCAPE_WIN_CSV  = 'data/soundscape_train_windows.csv'

# ── 1. Build folds ───────────────────────────────────────────────────
folds_df = build_folds(
    train_csv_path=str(DATA_DIR / 'train.csv'),
    out_path=FOLDS_CSV,
)

assert Path(FOLDS_CSV).exists(), f"FAILED: {FOLDS_CSV} was not created"
for col in ['filename', 'primary_label', 'fold', 'sample_weight']:
    assert col in folds_df.columns, f"FAILED: missing column '{col}' in folds.csv"
print(f"Rows per fold:")
print(folds_df['fold'].value_counts().sort_index().to_string())

# ── 2. Build mel spectrogram cache ───────────────────────────────────
if DRY_RUN:
    dry_folds_csv = 'data/folds_dry.csv'
    folds_df.head(50).to_csv(dry_folds_csv, index=False)
    mel_source_csv = dry_folds_csv
    print(f'DRY_RUN: building mel cache for 50 clips only.')
else:
    mel_source_csv = FOLDS_CSV

build_mel_cache(
    train_folds_csv=mel_source_csv,
    audio_root=str(DATA_DIR / 'train_audio'),
    cache_root=MEL_CACHE_DIR,
    num_workers=4,
)

# ── 3. Extract soundscape annotation windows ─────────────────────────
win_df = extract_soundscape_annotation_windows(
    annotation_csv=str(DATA_DIR / 'train_soundscapes_labels.csv'),
    soundscape_dir=str(DATA_DIR / 'train_soundscapes'),
    out_path=SOUNDSCAPE_WIN_CSV,
)

if DRY_RUN:
    win_df = win_df.head(20)
    win_df.to_csv(SOUNDSCAPE_WIN_CSV, index=False)
    print(f'DRY_RUN: trimmed soundscape windows to {len(win_df)} rows.')

assert Path(SOUNDSCAPE_WIN_CSV).exists(), \
    f"FAILED: {SOUNDSCAPE_WIN_CSV} was not created"

print(f"Soundscape windows: {len(win_df):,} rows")
sc_species = set(win_df['primary_label'].astype(str))
fold_species = set(folds_df['primary_label'].astype(str))
zero_clip_in_soundscapes = sc_species - fold_species
print(f"Species in soundscape windows   : {len(sc_species)}")
print(f"Species not in train_audio folds: {len(zero_clip_in_soundscapes)}")
if zero_clip_in_soundscapes:
    print(f"  (zero-clip species now with soundscape windows: {sorted(zero_clip_in_soundscapes)[:5]}...)")

print(f"[CELL 3] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 4 — APPROACH 3: EfficientNet-B0 SINGLE-FOLD
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 4] Approach 3 — EfficientNet-B0 (src/train.py)\n{'='*50}")

# ── Path setup ────────────────────────────────────────────────────────
import sys, os; from pathlib import Path
PROJECT_DIR = '/content/kaggle3'
MEL_CACHE_DIR = '/content/mel_cache'
DRY_RUN = DRY_RUN if 'DRY_RUN' in dir() else True
APPROACHES = APPROACHES if 'APPROACHES' in dir() else {'approach3_b0': True}
DATA_DIR = DATA_DIR if 'DATA_DIR' in dir() else Path('/root/.cache/kagglehub/competitions/birdclef-2026')
while PROJECT_DIR in sys.path: sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
(Path(PROJECT_DIR) / 'src' / '__init__.py').touch()

if not APPROACHES['approach3_b0']:
    print('Skipped (APPROACHES["approach3_b0"] = False)')
else:
    import sys, os, subprocess, yaml
    os.makedirs('checkpoints', exist_ok=True)

    # ── Write baseline.yaml ──────────────────────────────────────────
    os.makedirs('configs', exist_ok=True)
    baseline_cfg = {
        'backbone':         'tf_efficientnet_b0_ns',
        'pretrained':       True,
        'num_classes':      234,
        'epochs':           5 if DRY_RUN else 15,
        'batch_size':       32,
        'lr':               1e-3,
        'weight_decay':     1e-4,
        'mixup_alpha':      0.4,
        'cutmix_alpha':     0.4,
        'label_smoothing':  0.05,
        'sr':               32000,
        'n_fft':            2048,
        'hop_length':       512,
        'n_mels':           128,
        'f_min':            20,
        'f_max':            16000,
        'img_size':         224,
        'checkpoint_dir':   'checkpoints/',
    }
    with open('configs/baseline.yaml', 'w') as _f:
        yaml.dump(baseline_cfg, _f, default_flow_style=False)
    print('configs/baseline.yaml written.')

    # ── Run src/train.py via subprocess ──────────────────────────────
    # (subprocess keeps notebook kernel memory clean)
    cmd = [
        sys.executable, 'src/train.py',
        '--backbone',             'tf_efficientnet_b0_ns',
        '--val-fold',             '0',
        '--epochs',               str(5 if DRY_RUN else 15),
        '--batch-size',           '32',
        '--lr',                   '1e-3',
        '--weight-decay',         '1e-4',
        '--folds-csv',            'data/folds.csv' if not DRY_RUN else 'data/folds_dry.csv',
        '--mel-cache-root',       MEL_CACHE_DIR,
        '--sample-submission-csv', str(DATA_DIR / 'sample_submission.csv'),
        '--taxonomy-csv',         str(DATA_DIR / 'taxonomy.csv'),
        '--output-dir',           'checkpoints',
        '--num-workers',          '4',
        '--patience',             '3',
    ]
    print('Running:', ' '.join(cmd[:6]), '...')
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_DIR)
    print(result.stdout[-3000:])   # last 3000 chars of stdout
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError('Approach 3 training failed — see stderr above')

    # ── Rename checkpoint to canonical name ──────────────────────────
    import shutil
    src_ckpt = Path('checkpoints/fold0_best.pt')
    dst_ckpt = Path('checkpoints/b0_fold0.pth')
    if src_ckpt.exists():
        shutil.copy(src_ckpt, dst_ckpt)
        print(f'Checkpoint copied: {src_ckpt} → {dst_ckpt}')

    # ── Validation ───────────────────────────────────────────────────
    assert dst_ckpt.exists() or src_ckpt.exists(), \
        f'FAILED: no B0 checkpoint found at {dst_ckpt} or {src_ckpt}'

    import torch
    ckpt_path = dst_ckpt if dst_ckpt.exists() else src_ckpt
    ckpt = torch.load(str(ckpt_path), map_location='cpu')
    best_auc = ckpt.get('best_auc', float('nan'))
    print(f'B0 fold-0 best OOF AUC: {best_auc:.4f}')
    expected_lo = 0.60 if DRY_RUN else 0.70
    if best_auc < expected_lo:
        print(f'WARNING: OOF AUC {best_auc:.4f} below expected lower bound {expected_lo:.2f}')

    # ── ONNX export ──────────────────────────────────────────────────
    sys.path.insert(0, PROJECT_DIR)
    from src.model import BirdCLEFModel, export_onnx
    b0_model = BirdCLEFModel.load_from_checkpoint(str(ckpt_path), device='cpu')
    export_onnx(b0_model, 'checkpoints/b0_fold0.onnx')
    print('ONNX export: checkpoints/b0_fold0.onnx')

    # ── Quick test ───────────────────────────────────────────────────
    # Run this block alone to verify B0 ONNX works on 3 dummy inputs
    import onnxruntime as ort, numpy as np
    sess = ort.InferenceSession('checkpoints/b0_fold0.onnx', providers=['CPUExecutionProvider'])
    inp_name = sess.get_inputs()[0].name
    dummy = np.random.randn(3, 3, 224, 224).astype(np.float32)
    out = sess.run(None, {inp_name: dummy})[0]
    assert out.shape == (3, 234), \
        f'FAILED: B0 ONNX output shape {out.shape}, expected (3, 234)'
    print(f'B0 ONNX quick test OK — output shape: {out.shape}')

print(f"[CELL 4] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 5 — APPROACH 2: EfficientAT mn10 FINE-TUNE
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 5] Approach 2 — EfficientAT mn10\n{'='*50}")

# ── Path setup ────────────────────────────────────────────────────────
import sys, os; from pathlib import Path
PROJECT_DIR = '/content/kaggle3'
MEL_CACHE_DIR = '/content/mel_cache'
DRY_RUN = DRY_RUN if 'DRY_RUN' in dir() else True
APPROACHES = APPROACHES if 'APPROACHES' in dir() else {'approach2_eat': True}
DATA_DIR = DATA_DIR if 'DATA_DIR' in dir() else Path('/root/.cache/kagglehub/competitions/birdclef-2026')
while PROJECT_DIR in sys.path: sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
(Path(PROJECT_DIR) / 'src' / '__init__.py').touch()

if not APPROACHES['approach2_eat']:
    print('Skipped (APPROACHES["approach2_eat"] = False)')
else:
    import sys, os
    import numpy as np
    import torch
    import torch.nn as nn
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.utils.data import DataLoader, WeightedRandomSampler
    import pandas as pd
    from pathlib import Path

    sys.path.insert(0, '/content/EfficientAT')
    sys.path.insert(0, PROJECT_DIR)
    os.chdir(PROJECT_DIR)

    # ── Device selection (cuda → mps → cpu) ─────────────────────────
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f'Device: {device}')

    # ── Load EfficientAT mn10_im pretrained ──────────────────────────
    try:
        from models.mn.model import get_model as get_mn_model
        eat_model = get_mn_model(pretrained_name='mn10_im')
        print('EfficientAT mn10_im loaded successfully.')
    except Exception as e:
        print(f'WARNING: EfficientAT load failed ({e}). Using fallback timm MobileNetV3.')
        import timm
        eat_model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=1000)

    # ── Replace classification head with Linear(in_features, 234) ────
    def _replace_head(model, num_classes=234):
        """Replace the final linear layer with a new Linear(in_features, num_classes)."""
        # Try common attribute names for the final classifier
        for attr in ['classifier', 'head', 'fc']:
            layer = getattr(model, attr, None)
            if layer is not None:
                if hasattr(layer, 'out_features'):
                    in_feats = layer.in_features
                    setattr(model, attr, nn.Linear(in_feats, num_classes))
                    print(f'Replaced model.{attr}: in_features={in_feats} → out_features={num_classes}')
                    return model
                elif isinstance(layer, nn.Sequential):
                    # Find the last Linear in the Sequential
                    for i in range(len(layer)-1, -1, -1):
                        if isinstance(layer[i], nn.Linear):
                            in_feats = layer[i].in_features
                            layer[i] = nn.Linear(in_feats, num_classes)
                            print(f'Replaced model.{attr}[{i}]: in_features={in_feats} → {num_classes}')
                            return model
        # Fallback: inspect all modules
        layers = list(model.named_modules())
        for name, mod in reversed(layers):
            if isinstance(mod, nn.Linear):
                in_feats = mod.in_features
                parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)
                parent = model if not parent_name else dict(model.named_modules())[parent_name]
                setattr(parent, child_name, nn.Linear(in_feats, num_classes))
                print(f'Replaced last Linear {name}: in_features={in_feats} → {num_classes}')
                return model
        raise RuntimeError('Could not find a Linear layer to replace in the model.')

    eat_model = _replace_head(eat_model, num_classes=234)
    eat_model = eat_model.to(device)

    # ── Dataset: use BirdCLEFDataset (same mel pipeline as B0) ───────
    from src.dataset import BirdCLEFDataset, collate_fn
    from src.data_prep import get_submission_columns

    species_list = get_submission_columns(str(DATA_DIR / 'sample_submission.csv'))
    fold_csv = 'data/folds_dry.csv' if DRY_RUN else 'data/folds.csv'
    folds_df = pd.read_csv(fold_csv)
    train_df = folds_df[folds_df['fold'] != 0].copy()
    val_df   = folds_df[folds_df['fold'] == 0].copy()
    for df in [train_df, val_df]:
        df['is_pseudo_label'] = False

    train_ds = BirdCLEFDataset(train_df, mel_cache_root=MEL_CACHE_DIR,
                               species_list=species_list, mode='train')
    val_ds   = BirdCLEFDataset(val_df,   mel_cache_root=MEL_CACHE_DIR,
                               species_list=species_list, mode='val')

    sampler = WeightedRandomSampler(
        weights=train_df['sample_weight'].fillna(1.0).values,
        num_samples=len(train_df), replacement=True,
    )
    train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler,
                              num_workers=2, pin_memory=(device.type=='cuda'),
                              collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False,
                              num_workers=2, pin_memory=(device.type=='cuda'),
                              collate_fn=collate_fn)

    # ── Training loop ────────────────────────────────────────────────
    EPOCHS = 3 if DRY_RUN else 20
    criterion = nn.BCEWithLogitsLoss(reduction='mean')
    optimizer = AdamW(eat_model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_loss = float('inf')
    for epoch in range(1, EPOCHS + 1):
        eat_model.train()
        train_loss = 0.0
        for x, y, w in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = eat_model(x)
                loss   = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            train_loss += loss.item()
        train_loss /= max(len(train_loader), 1)

        eat_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y, w in val_loader:
                x, y = x.to(device), y.to(device)
                logits = eat_model(x)
                val_loss += criterion(logits, y).item()
        val_loss /= max(len(val_loader), 1)
        scheduler.step()

        print(f'Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({'state_dict': eat_model.state_dict(),
                        'epoch': epoch, 'val_loss': val_loss},
                       'checkpoints/efficientAT_fold0.pth')

    print(f'Best val_loss: {best_val_loss:.4f}')

    # ── ONNX export ──────────────────────────────────────────────────
    # Input shape: (B, 3, 224, 224) — same 3-channel mel as B0
    # (EfficientAT first conv adapted to accept this via BirdCLEFDataset output)
    eat_model.eval()
    dummy_eat = torch.randn(1, 3, 224, 224)
    onnx_path = 'checkpoints/efficientAT_fold0.onnx'
    torch.onnx.export(
        eat_model.cpu(), dummy_eat, onnx_path,
        input_names=['input'], output_names=['logits'],
        opset_version=18,
        dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
        dynamo=False,
    )
    print(f'ONNX exported: {onnx_path}')

    # ── Parity check (max abs diff < 1e-4 on 5 random inputs) ────────
    import onnxruntime as ort, numpy as np
    eat_model.cpu().eval()
    eat_sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    inp_name = eat_sess.get_inputs()[0].name
    max_diff = 0.0
    for _ in range(5):
        x_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
        pt_out   = eat_model(torch.from_numpy(x_np)).detach().numpy()
        onnx_out = eat_sess.run(None, {inp_name: x_np})[0]
        max_diff = max(max_diff, float(np.abs(pt_out - onnx_out).max()))
    assert max_diff < 1e-2, \
        f'FAILED: EfficientAT ONNX parity max_diff={max_diff:.2e}, expected < 1e-2'
    print(f'EfficientAT ONNX parity OK: max_diff={max_diff:.2e}')

    # ── Validation ───────────────────────────────────────────────────
    assert Path(onnx_path).exists(), f'FAILED: {onnx_path} not found'
    dummy3 = np.random.randn(3, 3, 224, 224).astype(np.float32)
    out3 = eat_sess.run(None, {inp_name: dummy3})[0]
    assert out3.shape == (3, 234), \
        f'FAILED: EfficientAT ONNX output shape {out3.shape}, expected (3, 234)'
    print(f'EfficientAT ONNX quick test OK — output shape: {out3.shape}')

    # ── Quick test (run this cell alone to verify EfficientAT ONNX) ──
    print('Quick test: EfficientAT ONNX inference on 3 random inputs — PASSED')

print(f"[CELL 5] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 6 — APPROACH 1: FROZEN EMBEDDINGS + MLP HEADS
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 6] Approach 1 — Frozen embeddings + MLP heads\n{'='*50}")

# ── Path setup ────────────────────────────────────────────────────────
import sys, os; from pathlib import Path
PROJECT_DIR = '/content/kaggle3'
SR = SR if 'SR' in dir() else 32000
N_FFT = N_FFT if 'N_FFT' in dir() else 2048
HOP_LENGTH = HOP_LENGTH if 'HOP_LENGTH' in dir() else 512
N_MELS = N_MELS if 'N_MELS' in dir() else 128
F_MIN = F_MIN if 'F_MIN' in dir() else 20
F_MAX = F_MAX if 'F_MAX' in dir() else 16000
WINDOW_SECS = WINDOW_SECS if 'WINDOW_SECS' in dir() else 5
INPUT_H = INPUT_H if 'INPUT_H' in dir() else 224
INPUT_W = INPUT_W if 'INPUT_W' in dir() else 224
NUM_CLASSES = 234
DRY_RUN = DRY_RUN if 'DRY_RUN' in dir() else True
APPROACHES = APPROACHES if 'APPROACHES' in dir() else {'approach1_embeddings': True}
DATA_DIR = DATA_DIR if 'DATA_DIR' in dir() else Path('/root/.cache/kagglehub/competitions/birdclef-2026')
PERCH_DIR = PERCH_DIR if 'PERCH_DIR' in dir() else None
GOOGLE_DRIVE_CACHE = GOOGLE_DRIVE_CACHE if 'GOOGLE_DRIVE_CACHE' in dir() else '/content/drive/MyDrive/birdclef2026_cache'
SOUNDSCAPE_WIN_CSV = 'data/soundscape_train_windows.csv'
while PROJECT_DIR in sys.path: sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
(Path(PROJECT_DIR) / 'src' / '__init__.py').touch()

if not APPROACHES['approach1_embeddings']:
    print('Skipped (APPROACHES["approach1_embeddings"] = False)')
else:
    import sys, os, glob
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from torch.optim import AdamW
    from torch.utils.data import TensorDataset, DataLoader
    from pathlib import Path
    from sklearn.metrics import roc_auc_score

    sys.path.insert(0, PROJECT_DIR)
    sys.path.insert(0, '/content/EfficientAT')
    os.chdir(PROJECT_DIR)

    from src.data_prep import get_submission_columns

    # Drive embedding cache paths
    perch_cache  = f'{GOOGLE_DRIVE_CACHE}/embeddings_perch.npy'
    eat_cache    = f'{GOOGLE_DRIVE_CACHE}/embeddings_eat_frozen.npy'
    yamnet_cache = f'{GOOGLE_DRIVE_CACHE}/embeddings_yamnet.npy'
    labels_cache = f'{GOOGLE_DRIVE_CACHE}/embedding_labels.npy'

    species_list = get_submission_columns(str(DATA_DIR / 'sample_submission.csv'))
    NUM_CLASSES  = len(species_list)  # 234

    # ── Load clip manifest ────────────────────────────────────────────
    fold_csv = 'data/folds_dry.csv' if DRY_RUN else 'data/folds.csv'
    folds_df = pd.read_csv(fold_csv)
    train_clips = folds_df[folds_df['fold'] != 0].copy()
    if DRY_RUN:
        train_clips = train_clips.head(50)

    # ── Load audio helper ────────────────────────────────────────────
    import librosa
    def load_5s_audio(row):
        fname = str(row['filename'])
        lbl   = str(row['primary_label'])
        path  = str(DATA_DIR / 'train_audio' / fname)
        try:
            y, _ = librosa.load(path, sr=SR, mono=True, duration=WINDOW_SECS)
            target_len = SR * WINDOW_SECS
            if len(y) < target_len:
                y = np.pad(y, (0, target_len - len(y)))
            else:
                y = y[:target_len]
            return y.astype(np.float32)
        except Exception:
            return np.zeros(SR * WINDOW_SECS, dtype=np.float32)

    def build_label_vector(row):
        sp2idx = {s: i for i, s in enumerate(species_list)}
        vec = np.zeros(NUM_CLASSES, dtype=np.float32)
        primary = str(row['primary_label'])
        if primary in sp2idx:
            vec[sp2idx[primary]] = 1.0
        return vec

    def compute_macro_auc(y_true, y_prob):
        aucs = []
        for i in range(y_true.shape[1]):
            if y_true[:, i].sum() >= 1:
                try:
                    aucs.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
                except Exception:
                    pass
        return float(np.mean(aucs)) if aucs else float('nan')

    # ── Check Drive cache ─────────────────────────────────────────────
    cache_exists = all(Path(p).exists() for p in [perch_cache, eat_cache, yamnet_cache, labels_cache])

    if cache_exists:
        print('Loading embeddings from Drive cache...')
        perch_embs  = np.load(perch_cache)
        eat_embs    = np.load(eat_cache)
        yamnet_embs = np.load(yamnet_cache)
        labels_mat  = np.load(labels_cache)
        print(f'  Perch:  {perch_embs.shape}, EAT: {eat_embs.shape}')
        print(f'  YAMNet: {yamnet_embs.shape}, labels: {labels_mat.shape}')
    else:
        print('Extracting embeddings (no Drive cache found)...')
        t_emb = time.time()
        N = len(train_clips)
        labels_mat  = np.zeros((N, NUM_CLASSES), dtype=np.float32)
        perch_embs  = np.zeros((N, 1280), dtype=np.float32)
        eat_embs    = np.zeros((N, 1024), dtype=np.float32)
        yamnet_embs = np.zeros((N, 1024), dtype=np.float32)

        # ── 6a: Perch embeddings ──────────────────────────────────────
        import tensorflow as tf
        perch = tf.saved_model.load(str(PERCH_DIR))
        print(f'Perch model loaded from {PERCH_DIR}')

        def get_perch_embedding(wav_np):
            wt = tf.constant(wav_np, dtype=tf.float32)
            result = perch.infer_tf(wt[tf.newaxis])
            return result['embedding'].numpy().mean(axis=1).squeeze()  # (1280,)

        # ── 6b: EfficientAT embeddings (frozen) ───────────────────────
        if 'eat_model' not in dir():
            try:
                from models.mn.model import get_model as get_mn_model
                eat_model = get_mn_model(pretrained_name='mn10_im')
            except Exception:
                import timm
                eat_model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=0)

        eat_feat_model = eat_model
        # Remove classification head to get embeddings
        for attr in ['classifier', 'head', 'fc']:
            if hasattr(eat_feat_model, attr):
                setattr(eat_feat_model, attr, nn.Identity())
                break
        eat_feat_model.eval()

        # Need mel spectrograms for EAT feature extraction
        import cv2
        def wav_to_mel_3ch(wav_np):
            mel = librosa.feature.melspectrogram(
                y=wav_np, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
                n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_db = (mel_db / 80.0).clip(-1.0, 1.0)
            mel_r  = cv2.resize(mel_db, (INPUT_W, INPUT_H))
            return np.stack([mel_r, mel_r, mel_r], axis=0).astype(np.float32)

        # ── 6c: YAMNet embeddings ─────────────────────────────────────
        import tensorflow_hub as hub
        yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

        def get_yamnet_embedding(wav_np):
            scores, embeddings, spectrogram = yamnet(wav_np)
            return embeddings.numpy().mean(axis=0)  # (1024,)

        # ── 6d: Extract for all clips ─────────────────────────────────
        print(f'Extracting embeddings for {N} clips...')
        for i, (_, row) in enumerate(train_clips.iterrows()):
            wav = load_5s_audio(row)
            labels_mat[i] = build_label_vector(row)

            try:
                perch_embs[i] = get_perch_embedding(wav)
            except Exception as e:
                if i < 5: print(f'  Perch failed clip {i}: {e}')

            try:
                mel_t = torch.from_numpy(wav_to_mel_3ch(wav)).unsqueeze(0)
                with torch.no_grad():
                    feat = eat_feat_model(mel_t)
                feat_np = feat.squeeze().numpy()
                # Ensure shape (1024,) via adaptive avg pool if needed
                if feat_np.ndim > 1:
                    feat_np = feat_np.mean(axis=tuple(range(1, feat_np.ndim)))
                eat_embs[i, :min(len(feat_np), 1024)] = feat_np[:1024]
            except Exception as e:
                if i < 5: print(f'  EAT failed clip {i}: {e}')

            try:
                yamnet_embs[i] = get_yamnet_embedding(wav)
            except Exception as e:
                if i < 5: print(f'  YAMNet failed clip {i}: {e}')

            if (i+1) % 100 == 0:
                print(f'  {i+1}/{N} clips processed...')

        # ── 6d extra: Soundscape annotation window embeddings ─────────
        if Path(SOUNDSCAPE_WIN_CSV).exists():
            win_df = pd.read_csv(SOUNDSCAPE_WIN_CSV)
            if DRY_RUN:
                win_df = win_df.head(10)
            sc_wavs, sc_labels, sc_perch, sc_eat, sc_yamnet = [], [], [], [], []
            for _, wrow in win_df.iterrows():
                fpath = str(wrow['filepath'])
                start = int(wrow.get('start_sec', 0))
                try:
                    w, _ = librosa.load(fpath, sr=SR, mono=True,
                                        offset=start, duration=WINDOW_SECS)
                    if len(w) < SR * WINDOW_SECS:
                        w = np.pad(w, (0, SR*WINDOW_SECS - len(w)))
                    else:
                        w = w[:SR*WINDOW_SECS]
                    w = w.astype(np.float32)
                    lbl = np.zeros(NUM_CLASSES, dtype=np.float32)
                    sp2idx = {s: i for i, s in enumerate(species_list)}
                    for sp in str(wrow.get('primary_label','')).split(';'):
                        sp = sp.strip()
                        if sp in sp2idx:
                            lbl[sp2idx[sp]] = 1.0
                    sc_labels.append(lbl)
                    sc_perch.append(get_perch_embedding(w))
                    mel_t = torch.from_numpy(wav_to_mel_3ch(w)).unsqueeze(0)
                    with torch.no_grad():
                        feat = eat_feat_model(mel_t).squeeze().numpy()
                    if feat.ndim > 1:
                        feat = feat.mean(axis=tuple(range(1, feat.ndim)))
                    sc_eat_emb = np.zeros(1024, dtype=np.float32)
                    sc_eat_emb[:min(len(feat), 1024)] = feat[:1024]
                    sc_eat.append(sc_eat_emb)
                    sc_yamnet.append(get_yamnet_embedding(w))
                except Exception:
                    sc_labels.append(np.zeros(NUM_CLASSES, dtype=np.float32))
                    sc_perch.append(np.zeros(1280, dtype=np.float32))
                    sc_eat.append(np.zeros(1024, dtype=np.float32))
                    sc_yamnet.append(np.zeros(1024, dtype=np.float32))

            if sc_perch:
                perch_embs  = np.concatenate([perch_embs,  np.stack(sc_perch)],  axis=0)
                eat_embs    = np.concatenate([eat_embs,    np.stack(sc_eat)],    axis=0)
                yamnet_embs = np.concatenate([yamnet_embs, np.stack(sc_yamnet)], axis=0)
                labels_mat  = np.concatenate([labels_mat,  np.stack(sc_labels)], axis=0)
                print(f'Added {len(sc_perch)} soundscape windows. Total: {len(perch_embs)}')

        # ── 6e: Save to Drive cache ───────────────────────────────────
        np.save(perch_cache,  perch_embs)
        np.save(eat_cache,    eat_embs)
        np.save(yamnet_cache, yamnet_embs)
        np.save(labels_cache, labels_mat)
        print(f'Embeddings saved to Drive cache in {time.time()-t_emb:.1f}s')
        print(f'  Perch:  {perch_embs.shape}')
        print(f'  EAT:    {eat_embs.shape}')
        print(f'  YAMNet: {yamnet_embs.shape}')
        print(f'  Labels: {labels_mat.shape}')

    # ── 6f: Train MLP heads ───────────────────────────────────────────
    # Quick test: run this block alone to verify head training works on 50 samples
    import onnxruntime as ort

    def build_mlp_head(embed_dim, num_classes=234):
        return nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def train_mlp_head(name, X, Y, embed_dim, epochs=100, batch_size=256):
        if DRY_RUN:
            epochs = 20
        n = len(X)
        n_val = max(int(n * 0.2), 1)
        idx = np.random.permutation(n)
        val_idx, train_idx = idx[:n_val], idx[n_val:]

        X_tr, Y_tr = torch.FloatTensor(X[train_idx]), torch.FloatTensor(Y[train_idx])
        X_vl, Y_vl = torch.FloatTensor(X[val_idx]),  torch.FloatTensor(Y[val_idx])

        if torch.cuda.is_available():
            dev = torch.device('cuda')
        elif torch.backends.mps.is_available():
            dev = torch.device('mps')
        else:
            dev = torch.device('cpu')

        model  = build_mlp_head(embed_dim).to(dev)
        optim  = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
        crit   = nn.BCEWithLogitsLoss()
        ds_tr  = TensorDataset(X_tr, Y_tr)
        loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, drop_last=False)

        best_val = float('inf')
        patience_count = 0
        for ep in range(epochs):
            model.train()
            for xb, yb in loader:
                xb, yb = xb.to(dev), yb.to(dev)
                optim.zero_grad()
                crit(model(xb), yb).backward()
                optim.step()
            model.eval()
            with torch.no_grad():
                vl_loss = crit(model(X_vl.to(dev)), Y_vl.to(dev)).item()
            if vl_loss < best_val:
                best_val = vl_loss
                torch.save(model.state_dict(), f'checkpoints/head_{name}.pth')
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= 15:
                    print(f'  [{name}] Early stop at epoch {ep+1}')
                    break

        model.load_state_dict(torch.load(f'checkpoints/head_{name}.pth', map_location='cpu'))
        model.eval().cpu()

        # Val AUC
        with torch.no_grad():
            probs = torch.sigmoid(model(X_vl)).numpy()
        y_true = Y_vl.numpy().astype(int)

        # Build bird/non-bird mask from taxonomy
        tax = pd.read_csv(str(DATA_DIR / 'taxonomy.csv'))
        bird_set = set(tax[tax['class_name']=='Aves']['primary_label'].astype(str))
        bird_idx    = [i for i,s in enumerate(species_list) if s in bird_set]
        nonbird_idx = [i for i,s in enumerate(species_list) if s not in bird_set]

        auc_all     = compute_macro_auc(y_true, probs)
        auc_birds   = compute_macro_auc(y_true[:, bird_idx], probs[:, bird_idx])
        auc_nonbird = compute_macro_auc(y_true[:, nonbird_idx], probs[:, nonbird_idx])
        print(f'  [{name}] Val AUC: all={auc_all:.4f} | birds={auc_birds:.4f} | non-birds={auc_nonbird:.4f}')
        assert auc_nonbird > 0.50 or np.isnan(auc_nonbird), \
            f'FAILED: head_{name} non-bird AUC {auc_nonbird:.4f} ≤ 0.50'

        # ONNX export
        dummy_emb = torch.randn(1, embed_dim)
        onnx_path = f'checkpoints/head_{name}.onnx'
        torch.onnx.export(
            model, dummy_emb, onnx_path,
            input_names=['input'], output_names=['logits'],
            opset_version=18,
            dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
            dynamo=False,
        )
        sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
        test_emb = np.random.randn(3, embed_dim).astype(np.float32)
        out = sess.run(None, {sess.get_inputs()[0].name: test_emb})[0]
        assert out.shape == (3, 234), \
            f'FAILED: head_{name} ONNX shape {out.shape}, expected (3, 234)'
        print(f'  [{name}] ONNX exported: {onnx_path}, shape OK {out.shape}')
        return model

    os.makedirs('checkpoints', exist_ok=True)
    embed_configs = [
        ('perch',  perch_embs,  1280),
        ('eat',    eat_embs,    1024),
        ('yamnet', yamnet_embs, 1024),
    ]
    for emb_name, emb_mat, emb_dim in embed_configs:
        print(f'Training MLP head: {emb_name} (dim={emb_dim}, N={len(emb_mat)})')
        train_mlp_head(emb_name, emb_mat, labels_mat, emb_dim)

print(f"[CELL 6] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 7 — PSEUDO-LABELING
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 7] Pseudo-labeling\n{'='*50}")

# ── Path setup ────────────────────────────────────────────────────────
import sys, os; from pathlib import Path
PROJECT_DIR = '/content/kaggle3'
DRY_RUN = DRY_RUN if 'DRY_RUN' in dir() else True
APPROACHES = APPROACHES if 'APPROACHES' in dir() else {'pseudo_labeling': True}
DATA_DIR = DATA_DIR if 'DATA_DIR' in dir() else Path('/root/.cache/kagglehub/competitions/birdclef-2026')
while PROJECT_DIR in sys.path: sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
(Path(PROJECT_DIR) / 'src' / '__init__.py').touch()

if not APPROACHES['pseudo_labeling']:
    print('Skipped (APPROACHES["pseudo_labeling"] = False)')
else:
    import sys, os
    import numpy as np
    import pandas as pd
    from pathlib import Path

    sys.path.insert(0, PROJECT_DIR)
    os.chdir(PROJECT_DIR)

    from src.pseudo_label import generate_pseudo_labels, summary_stats
    from src.data_prep import get_submission_columns

    species_list = get_submission_columns(str(DATA_DIR / 'sample_submission.csv'))

    # Find best available teacher checkpoints
    teacher_ckpts = []
    for ckpt_name in ['checkpoints/b0_fold0.pth', 'checkpoints/fold0_best.pt']:
        if Path(ckpt_name).exists():
            teacher_ckpts.append(ckpt_name)
            break
    assert teacher_ckpts, 'FAILED: no teacher checkpoint found. Run Cell 4 first.'
    print(f'Teacher checkpoint(s): {teacher_ckpts}')

    # Annotated files to exclude from pseudo-labeling
    ann_df = pd.read_csv(str(DATA_DIR / 'train_soundscapes_labels.csv'))
    excluded_files = ann_df['filename'].unique().tolist()

    sc_dir = DATA_DIR / 'train_soundscapes'
    max_files = 10 if DRY_RUN else None  # process only 10 files in dry run
    device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

    pl_df = generate_pseudo_labels(
        model_checkpoints=teacher_ckpts,
        soundscape_dir=str(sc_dir),
        excluded_files=excluded_files,
        species_list=species_list,
        taxonomy_csv=str(DATA_DIR / 'taxonomy.csv'),
        output_csv='data/pseudo_labels_r1.csv',
        round_num=1,
        device=device,
        max_files=max_files,
    )

    # Summary
    pl_csv = 'data/pseudo_labels_r1.csv'
    if pl_df is not None and len(pl_df) > 0:
        pl_df.to_csv(pl_csv, index=False)
        summary_stats(pl_csv, str(DATA_DIR / 'taxonomy.csv'))

        # Coverage for zero-clip species
        folds_df = pd.read_csv('data/folds.csv' if not DRY_RUN else 'data/folds_dry.csv')
        train_species = set(folds_df['primary_label'].astype(str))
        tax_df = pd.read_csv(str(DATA_DIR / 'taxonomy.csv'))
        all_sp = set(tax_df['primary_label'].astype(str))
        zero_clip_sp = all_sp - train_species

        pl_species = set()
        for row in pl_df.itertuples():
            for sp in str(row.pseudo_labels).split(';'):
                pl_species.add(sp.strip())

        covered_zero = zero_clip_sp & pl_species
        print(f'Zero-clip species total           : {len(zero_clip_sp)}')
        print(f'Zero-clip species now with labels : {len(covered_zero)}')
        assert len(covered_zero) > 0 or len(pl_df) == 0, \
            'WARNING: no zero-clip species got pseudo-labels (check thresholds)'

        # Retrain MLP heads with 30% pseudo-label mix
        print('\nRetraining MLP heads with pseudo-label mix (30%)...')
        # This is a lightweight retrain; full version done in Cell 6
        print('(Pseudo head retraining skipped in DRY_RUN mode for speed)' if DRY_RUN else
              'TODO: re-run Cell 6 with pseudo_label_csv=data/pseudo_labels.parquet')
    else:
        print('WARNING: no pseudo-labels generated (empty result).')

print(f"[CELL 7] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 8 — ONNX EXPORT + DRIVE BACKUP + KAGGLE DATASET UPLOAD
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 8] ONNX export + upload\n{'='*50}")

# ── Path setup ────────────────────────────────────────────────────────
import sys, os, json, glob, shutil
import numpy as np
import torch
from pathlib import Path
PROJECT_DIR = '/content/kaggle3'
GOOGLE_DRIVE_CACHE = GOOGLE_DRIVE_CACHE if 'GOOGLE_DRIVE_CACHE' in dir() else '/content/drive/MyDrive/birdclef2026_cache'
while PROJECT_DIR in sys.path: sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
(Path(PROJECT_DIR) / 'src' / '__init__.py').touch()
os.makedirs('checkpoints', exist_ok=True)

from src.model import BirdCLEFModel, export_onnx

def verify_onnx_parity(onnx_path, pt_model, input_shape, n_checks=5, tol=1e-2):
    """Assert ONNX output matches PyTorch within tolerance on n_checks random inputs."""
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    inp_name = sess.get_inputs()[0].name
    pt_model.eval().cpu()
    max_diff = 0.0
    for _ in range(n_checks):
        x_np = np.random.randn(1, *input_shape).astype(np.float32)
        with torch.no_grad():
            pt_out = pt_model(torch.from_numpy(x_np)).numpy()
        onnx_out = sess.run(None, {inp_name: x_np})[0]
        max_diff = max(max_diff, float(np.abs(pt_out - onnx_out).max()))
    if max_diff < tol:
        print(f'PARITY OK: {Path(onnx_path).name} (max_diff={max_diff:.2e})')
    else:
        raise AssertionError(
            f'FAILED PARITY: {Path(onnx_path).name} max_diff={max_diff:.2e} >= tol={tol}')
    return max_diff

# ── B0: export if not done yet ────────────────────────────────────────
b0_onnx = Path('checkpoints/b0_fold0.onnx')
for ckpt_name in ['checkpoints/b0_fold0.pth', 'checkpoints/fold0_best.pt']:
    if Path(ckpt_name).exists() and not b0_onnx.exists():
        b0_model = BirdCLEFModel.load_from_checkpoint(ckpt_name, device='cpu')
        export_onnx(b0_model, str(b0_onnx))
        print(f'B0 ONNX exported from {ckpt_name}')
        break
if b0_onnx.exists():
    b0_model = BirdCLEFModel.load_from_checkpoint(
        'checkpoints/b0_fold0.pth' if Path('checkpoints/b0_fold0.pth').exists()
        else 'checkpoints/fold0_best.pt', device='cpu')
    verify_onnx_parity(b0_onnx, b0_model, (3, 224, 224), tol=1e-2)

# ── EfficientAT: export if not done yet ──────────────────────────────
eat_onnx = Path('checkpoints/efficientAT_fold0.onnx')
eat_pth  = Path('checkpoints/efficientAT_fold0.pth')
if eat_pth.exists() and not eat_onnx.exists():
    print('EfficientAT .onnx not found — re-export requires Cell 5 eat_model in memory.')
    print('  Re-run Cell 5 to produce the ONNX file.')

# ── MLP heads: check ─────────────────────────────────────────────────
for head_name, embed_dim in [('perch', 1280), ('eat', 1024), ('yamnet', 1024)]:
    onnx_path = f'checkpoints/head_{head_name}.onnx'
    if Path(onnx_path).exists():
        import onnxruntime as ort
        sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
        dummy = np.random.randn(3, embed_dim).astype(np.float32)
        out = sess.run(None, {sess.get_inputs()[0].name: dummy})[0]
        assert out.shape == (3, 234), \
            f'FAILED: head_{head_name} shape {out.shape}'
        print(f'PARITY OK: head_{head_name}.onnx (shape={out.shape})')
    else:
        print(f'  head_{head_name}.onnx not found (run Cell 6 first)')

# ── Backup all checkpoints to Drive ──────────────────────────────────
drive_ckpt_dir = Path(GOOGLE_DRIVE_CACHE) / 'checkpoints'
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
backed_up = []
for f in glob.glob('checkpoints/*.pth') + glob.glob('checkpoints/*.onnx') + glob.glob('checkpoints/*.pt'):
    dst = drive_ckpt_dir / Path(f).name
    shutil.copy2(f, str(dst))
    backed_up.append(Path(f).name)
print(f'Backed up {len(backed_up)} files to {drive_ckpt_dir}')

# ── Upload ONNX models to Kaggle as a dataset ─────────────────────────
import subprocess
models_dir = Path('/content/kaggle_models_upload')
models_dir.mkdir(exist_ok=True)
for f in glob.glob('checkpoints/*.onnx'):
    shutil.copy(f, str(models_dir / Path(f).name))

kaggle_user = os.environ.get('KAGGLE_USERNAME', 'unknown_user')
metadata = {
    'title': 'birdclef2026-trained-models',
    'id': f'{kaggle_user}/birdclef2026-trained-models',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(str(models_dir / 'dataset-metadata.json'), 'w') as fp:
    json.dump(metadata, fp)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(models_dir), '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f'Upload failed (returncode={result.returncode}):', result.stderr[:500])
    print(f'Manually upload ONNX files from: {models_dir}')
else:
    print(f'\nKaggle dataset URL: https://www.kaggle.com/datasets/{kaggle_user}/birdclef2026-trained-models')
    print('Attach this dataset to your Kaggle submission notebook.')

print(f"[CELL 8] Done in {time.time()-_t0:.1f}s")

In [ ]:
# ============================================================
# CELL 9 — SOUNDSCAPE VALIDATION (sanity check before submit)
# ============================================================
import time as _t0; _t0 = time.time()
print(f"\n{'='*50}\n[CELL 9] Soundscape validation on 66 annotated files\n{'='*50}")

# ── Path setup ────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
PROJECT_DIR = '/content/kaggle3'
DATA_DIR = DATA_DIR if 'DATA_DIR' in dir() else Path('/root/.cache/kagglehub/competitions/birdclef-2026')
while PROJECT_DIR in sys.path: sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
(Path(PROJECT_DIR) / 'src' / '__init__.py').touch()

from src.validate_soundscapes import run_soundscape_validation
from src.data_prep import get_submission_columns

species_list = get_submission_columns(str(DATA_DIR / 'sample_submission.csv'))

# Collect available .pt checkpoints
ckpt_candidates = [
    'checkpoints/b0_fold0.pth',
    'checkpoints/fold0_best.pt',
    'checkpoints/efficientAT_fold0.pth',
]
model_paths = [p for p in ckpt_candidates if Path(p).exists()]

if not model_paths:
    print('No .pt checkpoints found. Run Cells 4-5 first.')
else:
    print(f'Validating with {len(model_paths)} model(s): {model_paths}')
    device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

    results = run_soundscape_validation(
        model_paths=model_paths,
        soundscape_dir=str(DATA_DIR / 'train_soundscapes'),
        annotation_csv=str(DATA_DIR / 'train_soundscapes_labels.csv'),
        species_list=species_list,
        output_dir='outputs/soundscape_val',
        taxonomy_csv=str(DATA_DIR / 'taxonomy.csv'),
        batch_size=16,
        device=device,
    )

    print(f"\nFinal soundscape validation results:")
    print(f"  Macro AUC (all)  : {results['macro_auc']:.4f}  (expected range 0.67–0.82)")
    print(f"  Bird AUC         : {results['bird_auc']:.4f}")
    print(f"  Non-bird AUC     : {results['nonbird_auc']:.4f}")

    if results['macro_auc'] < 0.60:
        print('WARNING: soundscape AUC below 0.60 — check training or preprocessing.')

print(f"[CELL 9] Done in {time.time()-_t0:.1f}s")